# Rubric USEMO Lab

This notebook tests the rubric flow on the USEMO 2020 P1 fixture:

1. Load and validate the rubric.
2. Exercise deterministic scoring on known judgment trees.
3. Build grading prompts for candidate solutions.
4. Optionally generate candidate solutions with a weaker model.
5. Optionally grade those candidates with the rubric prompt and compare computed scores against expected scores.

Model calls are disabled by default. Set `RUN_MODEL_CALLS = True` only when you want to spend API calls.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from textwrap import dedent
from dotenv import load_dotenv

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'rubric_arena' / 'rubric_grading.py').exists():
            return candidate
    raise RuntimeError(
        'Could not find the rubric-arena repo root. Start Jupyter from the repo, '
        'or set REPO_ROOT manually in this cell.'
    )

REPO_ROOT = find_repo_root(Path.cwd()).resolve()
load_dotenv(REPO_ROOT / '.env')
print('repo root:', REPO_ROOT)
print('anthropic key loaded:', bool(os.environ.get('ANTHROPIC_API_KEY')))

spec = importlib.util.spec_from_file_location(
    'rubric_grading', REPO_ROOT / 'src' / 'rubric_arena' / 'rubric_grading.py'
)
rubric_grading = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = rubric_grading
assert spec.loader is not None
spec.loader.exec_module(rubric_grading)

RUBRIC_PATH = REPO_ROOT / 'data/usemo_2020/rubrics/usemo_2020_p1.rubric.json'
OUTPUT_ROOT = REPO_ROOT / 'data/usemo_2020'
CANDIDATE_DIR = OUTPUT_ROOT / 'candidate_solutions' / 'usemo_2020_p1'
GRADING_DIR = OUTPUT_ROOT / 'grading_runs' / 'usemo_2020_p1'
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
GRADING_DIR.mkdir(parents=True, exist_ok=True)

rubric = json.loads(RUBRIC_PATH.read_text())
validation = rubric_grading.validate_rubric(rubric)
print('loaded', RUBRIC_PATH)
print('warnings', validation.warnings)


In [ ]:
PROBLEM = dedent('''
Which positive integers can be written in the form

    (lcm(x,y) + lcm(y,z)) / lcm(x,z)

for positive integers x,y,z?
''').strip()

REFERENCE_SOLUTION = dedent('''
The possible values are exactly the even positive integers.

For every even k=2n, take x=1, y=n, z=1. Then lcm(x,y)=n,
lcm(y,z)=n, and lcm(x,z)=1, so the expression equals 2n.

For the converse, suppose the value k is odd. One official route uses
2-adic valuations. From

    -k*lcm(x,z) + lcm(x,y) + lcm(y,z) = 0,

the three terms have 2-adic valuations determined by pairwise maxima of
v2(x), v2(y), v2(z). Since k is odd, the valuation of k*lcm(x,z) is just
the valuation of lcm(x,z). The valuation analysis rules out all odd k.
Alternative official routes use pairwise-gcd factorization or a general
prime-adic lcm divisibility claim.
''').strip()


## Known Judgment Fixtures

These are not model outputs. They are hand-authored judgments that test whether the rubric validator and scorer behave correctly.


In [ ]:
def complete_judgment():
    return {
        'id': 'usemo_2020_p1',
        'reasoning': 'The paper gives a complete proof of both directions.',
        'selected': 'usemo_2020_p1.complete',
        'children': [
            {
                'id': 'usemo_2020_p1.complete',
                'reasoning': 'Both required complete-solution conditions are satisfied.',
                'satisfied': True,
                'children': [
                    {'id': 'usemo_2020_p1.complete.even_construction', 'reasoning': 'It constructs x=1,z=1,y=n for k=2n.', 'satisfied': True},
                    {'id': 'usemo_2020_p1.complete.odd_impossible', 'reasoning': 'It proves the odd case cannot occur.', 'satisfied': True},
                ],
            }
        ],
    }


def partial_v2_judgment():
    return {
        'id': 'usemo_2020_p1',
        'reasoning': 'The paper has even construction and at least one substantial valuation case but is incomplete.',
        'selected': 'usemo_2020_p1.partial_v2',
        'children': [
            {
                'id': 'usemo_2020_p1.partial_v2',
                'reasoning': 'It earns both additive items in the v2 partial route.',
                'children': [
                    {'id': 'usemo_2020_p1.partial_v2.even_construction', 'reasoning': 'It gives x=1,z=1,y=n for k=2n.', 'satisfied': True},
                    {'id': 'usemo_2020_p1.partial_v2.substantial_case', 'reasoning': 'It handles the case where v2(y) is strictly maximal.', 'satisfied': True},
                ],
            }
        ],
    }


def no_progress_judgment():
    return {
        'id': 'usemo_2020_p1',
        'reasoning': 'None of the above applies; the paper only states the answer.',
        'selected': 'usemo_2020_p1.no_progress',
        'children': [{'id': 'usemo_2020_p1.no_progress', 'reasoning': 'The paper has no score-bearing construction or proof.', 'satisfied': False}],
    }

for name, judgment, expected in [
    ('complete', complete_judgment(), 7),
    ('partial_v2', partial_v2_judgment(), 2),
    ('no_progress', no_progress_judgment(), 0),
]:
    score = rubric_grading.compute_score(rubric, judgment)
    print(name, score, 'expected', expected)
    assert score == expected


## Candidate Solution Fixtures

These candidate solutions let us test prompt formatting and end-to-end score computation before involving external models.


In [ ]:
CANDIDATE_FIXTURES = [
    {
        'id': 'answer_only_even',
        'expected_score': 0,
        'expected_regime': 'usemo_2020_p1.no_progress',
        'candidate_solution': 'The answer is exactly the even positive integers.',
    },
    {
        'id': 'construction_plus_one_v2_case',
        'expected_score': 2,
        'expected_regime': 'usemo_2020_p1.partial_v2',
        'candidate_solution': dedent('''
        I claim the answer is the even positive integers.

        First, every even integer works. Let k=2n. Choose x=1, z=1, and y=n.
        Then lcm(x,y)=n, lcm(y,z)=n, and lcm(x,z)=1, so the expression is 2n.

        Now suppose k is odd. Write A=v2(x), B=v2(y), and C=v2(z).
        I will look only at the case where B is strictly larger than both A and C.
        Then v2(lcm(x,y))=B and v2(lcm(y,z))=B, while v2(k*lcm(x,z))=max(A,C),
        which is smaller. This case is impossible. I have not checked the other cases.
        ''').strip(),
    },
]

for fixture in CANDIDATE_FIXTURES:
    out = CANDIDATE_DIR / f"{fixture['id']}.json"
    out.write_text(json.dumps(fixture, indent=2, ensure_ascii=False) + '\n')
    print('wrote', out)


In [ ]:
fixture = CANDIDATE_FIXTURES[1]
prompt = rubric_grading.build_grading_prompt(
    problem=PROBLEM,
    reference_solution=REFERENCE_SOLUTION,
    candidate_solution=fixture['candidate_solution'],
    rubric=rubric,
)
print(prompt[:2000])
print('\n... prompt chars:', len(prompt))


## Optional: Generate Candidate Solutions with a Weaker Model

This section writes candidate solution JSON files under:

```text
data/usemo_2020/candidate_solutions/usemo_2020_p1/
```

Set `RUN_MODEL_CALLS = True` and configure `ANTHROPIC_API_KEY` in the environment before running.


In [ ]:
RUN_MODEL_CALLS = False
CANDIDATE_MODEL = 'claude-sonnet-4-6'  # Change to a specific Sonnet 4.5 endpoint if that is the desired test model.
GRADER_MODEL = 'claude-sonnet-4-6'


In [ ]:
def call_anthropic_text(prompt: str, *, model: str, max_tokens: int = 4096) -> str:
    import anthropic

    client = anthropic.Anthropic()
    message = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=0.7,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return ''.join(
        block.text for block in message.content if getattr(block, 'type', None) == 'text'
    )


def generate_candidate_solution(target: str) -> str:
    prompt = f'''
Solve the following olympiad problem. Write a contestant-style solution.
Target behavior for evaluation: {target}

Problem:
{PROBLEM}
'''.strip()
    return call_anthropic_text(prompt, model=CANDIDATE_MODEL, max_tokens=4096)

if RUN_MODEL_CALLS:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY is required'
    generated_specs = [
        ('model_answer_only', 0, 'give only the final answer with almost no proof'),
        ('model_partial_v2', 2, 'include the even construction and one valuation case, but leave the proof incomplete'),
        ('model_full_attempt', 7, 'try to give a complete proof'),
    ]
    for candidate_id, expected_score, target in generated_specs:
        candidate_solution = generate_candidate_solution(target)
        record = {
            'id': candidate_id,
            'problem_id': 'usemo_2020_p1',
            'model': CANDIDATE_MODEL,
            'created_at': datetime.now(timezone.utc).isoformat(),
            'expected_score': expected_score,
            'generation_target': target,
            'candidate_solution': candidate_solution,
        }
        path = CANDIDATE_DIR / f'{candidate_id}.json'
        path.write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n')
        print('wrote', path)
else:
    print('model calls disabled')


## Optional: Grade Candidate Solutions with the Rubric

This sends the rubric grading prompt to a model, parses the judgment, computes the score deterministically, and writes result JSON under:

```text
data/usemo_2020/grading_runs/usemo_2020_p1/
```


In [ ]:
def grade_candidate_with_model(candidate_record: dict) -> dict:
    prompt = rubric_grading.build_grading_prompt(
        problem=PROBLEM,
        reference_solution=REFERENCE_SOLUTION,
        candidate_solution=candidate_record['candidate_solution'],
        rubric=rubric,
    )
    raw = call_anthropic_text(prompt, model=GRADER_MODEL, max_tokens=4096)
    result = rubric_grading.grade_from_model_output(rubric=rubric, raw_model_output=raw)
    return {
        'candidate_id': candidate_record['id'],
        'problem_id': 'usemo_2020_p1',
        'rubric_path': str(RUBRIC_PATH.relative_to(REPO_ROOT)),
        'grader_model': GRADER_MODEL,
        'expected_score': candidate_record.get('expected_score'),
        **result,
    }

if RUN_MODEL_CALLS:
    for candidate_path in sorted(CANDIDATE_DIR.glob('*.json')):
        candidate_record = json.loads(candidate_path.read_text())
        result = grade_candidate_with_model(candidate_record)
        out = GRADING_DIR / f"{candidate_path.stem}.{GRADER_MODEL}.result.json"
        out.write_text(json.dumps(result, indent=2, ensure_ascii=False) + '\n')
        print(candidate_path.name, 'expected', result['expected_score'], 'computed', result['computed_score'], '->', out)
else:
    print('model calls disabled')


## Optional: Generate a Rubric from a Source Grading Scheme

This is the missing rubric-generation leg of the experiment. It sends the original grading scheme to an LLM, asks for rubric JSON, validates the result with `rubric_from_model_output`, and writes it to the dataset-local rubric directory.

This cell is disabled unless `RUN_MODEL_CALLS = True`.


In [ ]:
def generate_rubric_with_model(
    *,
    problem_id: str,
    problem: str,
    source_grading_scheme,
    sample_solution: str = '',
    max_points: int = 7,
    model: str = GRADER_MODEL,
) -> dict:
    prompt = rubric_grading.build_rubric_generation_prompt(
        problem_id=problem_id,
        problem=problem,
        source_grading_scheme=source_grading_scheme,
        sample_solution=sample_solution,
        max_points=max_points,
    )
    raw = call_anthropic_text(prompt, model=model, max_tokens=8192)
    generated_rubric = rubric_grading.rubric_from_model_output(raw)
    return {
        'rubric': generated_rubric,
        'raw_model_output': raw,
        'prompt': prompt,
    }

if RUN_MODEL_CALLS:
    result = generate_rubric_with_model(
        problem_id='usemo_2020_p1_generated',
        problem=PROBLEM,
        source_grading_scheme=json.loads(RUBRIC_PATH.read_text()),
        sample_solution=REFERENCE_SOLUTION,
        max_points=7,
    )
    out = RUBRIC_PATH.parent / 'usemo_2020_p1.generated.rubric.json'
    out.write_text(json.dumps(result['rubric'], indent=2, ensure_ascii=False) + '\n')
    print('wrote', out)
else:
    print('model calls disabled')


## Optional: MathArena USAMO 2026

This section expands the lab to MathArena's USAMO 2026 datasets:

- `MathArena/usamo_2026` contains problems, sample solutions, and source grading schemes.
- `MathArena/usamo_2026_outputs` contains model answers and judge scores/details.

It loads rows with Hugging Face `datasets`, creates rubrics from MathArena grading schemes, then grades MathArena model outputs with our rubric prompt/parser/scorer.


In [ ]:
MATHARENA_PROBLEMS_REPO = 'MathArena/usamo_2026'
MATHARENA_OUTPUTS_REPO = 'MathArena/usamo_2026_outputs'
MATHARENA_PROBLEM_IDX = 2
MATHARENA_OUTPUT_MODEL_NAME = 'Gemini 3.1 Pro Preview'
MATHARENA_OUTPUT_ROW_LIMIT = 2

MATHARENA_ROOT = REPO_ROOT / 'data/matharena_usamo_2026'
MATHARENA_RUBRIC_DIR = MATHARENA_ROOT / 'rubrics'
MATHARENA_CANDIDATE_DIR = MATHARENA_ROOT / 'candidate_solutions'
MATHARENA_GRADING_DIR = MATHARENA_ROOT / 'grading_runs'
for directory in [MATHARENA_RUBRIC_DIR, MATHARENA_CANDIDATE_DIR, MATHARENA_GRADING_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_matharena_rows():
    from datasets import load_dataset

    problems = load_dataset(MATHARENA_PROBLEMS_REPO, split='train')
    outputs = load_dataset(MATHARENA_OUTPUTS_REPO, split='train')
    problem_rows = [row for row in problems if int(row['problem_idx']) == int(MATHARENA_PROBLEM_IDX)]
    output_rows = [
        row for row in outputs
        if int(row['problem_idx']) == int(MATHARENA_PROBLEM_IDX)
        and row.get('model_name') == MATHARENA_OUTPUT_MODEL_NAME
    ]
    return problem_rows, output_rows[:MATHARENA_OUTPUT_ROW_LIMIT]

if RUN_MODEL_CALLS:
    matharena_problem_rows, matharena_output_rows = load_matharena_rows()
    assert matharena_problem_rows, f'No MathArena problem row for problem_idx={MATHARENA_PROBLEM_IDX}'
    assert matharena_output_rows, f'No MathArena output rows for {MATHARENA_OUTPUT_MODEL_NAME}'
    print('problem rows', len(matharena_problem_rows))
    print('output rows selected', len(matharena_output_rows))
else:
    print('model calls disabled; skipping Hugging Face dataset load')


In [ ]:
def normalize_matharena_grading_scheme(value):
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value

if RUN_MODEL_CALLS:
    source_problem = matharena_problem_rows[0]
    problem_id = f"matharena_usamo_2026_p{int(source_problem['problem_idx'])}"
    generation_result = generate_rubric_with_model(
        problem_id=problem_id,
        problem=source_problem['problem'],
        source_grading_scheme=normalize_matharena_grading_scheme(source_problem['grading_scheme']),
        sample_solution=source_problem.get('sample_solution') or '',
        max_points=int(source_problem.get('points') or 7),
        model=GRADER_MODEL,
    )
    matharena_rubric = generation_result['rubric']
    rubric_out = MATHARENA_RUBRIC_DIR / f'{problem_id}.rubric.json'
    rubric_out.write_text(json.dumps(matharena_rubric, indent=2, ensure_ascii=False) + '\n')
    print('wrote', rubric_out)
else:
    print('model calls disabled')


In [ ]:
def write_matharena_candidate(row: dict) -> Path:
    problem_id = f"matharena_usamo_2026_p{int(row['problem_idx'])}"
    safe_model = row['model_name'].lower().replace(' ', '_').replace('.', '').replace('/', '_')
    candidate_id = f"{problem_id}.{safe_model}.{row['idx_answer']}"
    record = {
        'id': candidate_id,
        'problem_id': problem_id,
        'model_name': row['model_name'],
        'model_config': row.get('model_config'),
        'idx_answer': row.get('idx_answer'),
        'candidate_solution': row.get('answer') or '',
        'ground_truth_score': row.get('points_judge_1'),
        'ground_truth_max_points': row.get('max_points_judge_1'),
        'matharena_grading_details': row.get('grading_details_judge_1'),
    }
    out = MATHARENA_CANDIDATE_DIR / f'{candidate_id}.json'
    out.write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n')
    return out

if RUN_MODEL_CALLS:
    candidate_paths = [write_matharena_candidate(row) for row in matharena_output_rows]
    for path in candidate_paths:
        print('wrote', path)
else:
    print('model calls disabled')


In [ ]:
if RUN_MODEL_CALLS:
    for candidate_path in candidate_paths:
        candidate = json.loads(candidate_path.read_text())
        prompt = rubric_grading.build_grading_prompt(
            problem=source_problem['problem'],
            reference_solution=source_problem.get('sample_solution') or '',
            candidate_solution=candidate['candidate_solution'],
            rubric=matharena_rubric,
        )
        raw = call_anthropic_text(prompt, model=GRADER_MODEL, max_tokens=8192)
        result = rubric_grading.grade_from_model_output(rubric=matharena_rubric, raw_model_output=raw)
        output = {
            'candidate_id': candidate['id'],
            'problem_id': candidate['problem_id'],
            'grader_model': GRADER_MODEL,
            'ground_truth_score': candidate.get('ground_truth_score'),
            **result,
        }
        out = MATHARENA_GRADING_DIR / f"{candidate_path.stem}.{GRADER_MODEL}.result.json"
        out.write_text(json.dumps(output, indent=2, ensure_ascii=False) + '\n')
        print(candidate['id'], 'ground truth', output['ground_truth_score'], 'computed', output['computed_score'], '->', out)
else:
    print('model calls disabled')


## Local Result Inspection

After model grading runs, use this cell to compare expected and computed scores.


In [ ]:
rows = []
for path in sorted(GRADING_DIR.glob('*.result.json')):
    result = json.loads(path.read_text())
    rows.append({
        'file': path.name,
        'candidate_id': result.get('candidate_id'),
        'expected_score': result.get('expected_score'),
        'computed_score': result.get('computed_score'),
        'score_consistent': result.get('score_consistent'),
    })
rows
